<a href="https://colab.research.google.com/github/jaraiudigital/WebScrapingFalabella/blob/main/webscrapingfalabella.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ==============================================================================================
# 1. COMANDOS DE INSTALACIÓN EN EL SISTEMA OPERATIVO Y PIP (EXCLUSIVO PARA GOOGLE COLAB)
# ==============================================================================================

# Descarga silenciosa (-q) del paquete instalador ejecutable oficial de Google Chrome para Linux (servidor Colab)
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb

# Instala el paquete .deb descargado de Chrome y resuelve automáticamente cualquier dependencia faltante (-f)
!dpkg -i google-chrome-stable_current_amd64.deb || apt-get install -f -y > /dev/null

# Instala las librerías necesarias de Python: Selenium (automatización), webdriver-manager (gestión de driver), BeautifulSoup4 y Pandas
!pip install selenium webdriver-manager beautifulsoup4 pandas > /dev/null


# ==============================================================================================
# 2. IMPORTACIÓN DE MÓDULOS Y LIBRERÍAS DE PYTHON
# ==============================================================================================

import json  # Módulo nativo para decodificar, procesar y manipular cadenas de texto en formato JSON
import time  # Módulo nativo para gestionar pausas de tiempo y retardo de ejecución (delays)
import pandas as pd  # Librería fundamental para el tratamiento, limpieza y exportación de estructuras de datos (DataFrames)
from bs4 import BeautifulSoup  # Herramienta para analizar sintácticamente (parser) el código fuente HTML obtenido
from selenium import webdriver  # Módulo principal para controlar y automatizar la navegación web real
from selenium.webdriver.chrome.service import Service  # Clase para gestionar el servicio en segundo plano del ejecutable ChromeDriver
from selenium.webdriver.chrome.options import Options  # Clase para definir argumentos y configuraciones de arranque del navegador Chrome
from webdriver_manager.chrome import ChromeDriverManager  # Herramienta que autodetecta e instala la versión compatible de ChromeDriver

# Control de entorno: detecta si la ejecución se realiza dentro de Google Colab para habilitar la descarga automática
try:
    from google.colab import files  # Permite disparar la descarga directa del archivo generado al computador del usuario
    EN_COLAB = True  # Bandera booleana en True si estamos dentro de Google Colab
except ImportError:
    EN_COLAB = False  # Bandera booleana en False si se ejecuta localmente (Jupyter Notebook / VS Code)

(Reading database ... 123192 files and directories currently installed.)
Preparing to unpack google-chrome-stable_current_amd64.deb ...
Unpacking google-chrome-stable (156.0.8078.47-1) over (156.0.8078.47-1) ...
Setting up google-chrome-stable (156.0.8078.47-1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [4]:
# ==============================================================================================
# 3. CONFIGURACIÓN DEL NAVEGADOR CHROME EN MODO SILENCIOSO (HEADLESS)
# ==============================================================================================

# Instanciamos el objeto de opciones para personalizar el comportamiento del navegador Chrome
options = Options()

# Activa el modo "Headless" moderno (ejecuta el navegador en segundo plano sin ventana gráfica, obligatorio en Colab)
options.add_argument('--headless=new')

# Desactiva la protección Sandbox de Linux (previene fallos de permisos dentro del contenedor virtual de Colab)
options.add_argument('--no-sandbox')

# Evita la restricción de memoria compartida temporal del sistema Linux (/dev/shm) evitando que el navegador colapse
options.add_argument('--disable-dev-shm-usage')

# Desactiva el uso de aceleración gráfica por hardware (GPU), innecesario en servidores sin pantalla
options.add_argument('--disable-gpu')

# Fuerza una resolución de pantalla de escritorio fija (1920x1080) para asegurar la carga completa de elementos
options.add_argument('--window-size=1920,1080')

# Define un encabezado de simulación (User-Agent) idéntico al de un usuario humano navegando en Windows para evitar bloqueos WAF
options.add_argument(
    'user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'
)


# ==============================================================================================
# 4. INICIALIZACIÓN Y CONFIGURACIÓN DEL NAVEGADOR AUTOMATIZADO
# ==============================================================================================

# Descarga automáticamente la versión exacta de ChromeDriver requerida e inicia la sesión del navegador automatizado
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# Lista vacía global donde se irán acumulando los diccionarios con la información de cada bicicleta extraída
todos_los_productos = []

# Definimos la cantidad de páginas del catálogo que deseamos recorrer secuencialmente
paginas_a_descargar = 5

print("Iniciando la extracción automatizada de Bicicletas en Falabella...\n")


# ==============================================================================================
# 5. BUCLE PRINCIPAL DE EXTRACCIÓN Y RECORRIDO DE PÁGINAS
# ==============================================================================================

try:
    # Recorremos un rango numérico secuencial desde la página 1 hasta la 5
    for pagina in range(1, paginas_a_descargar + 1):

        # Construimos la URL objetivo inyectando dinámicamente el número de página actual en el parámetro ?page=
        url_dinamica = (
            f"https://www.falabella.com.co/falabella-co/category/cat50516/Bicicletas"
            f"?page={pagina}"
        )

        # Imprimimos en consola el estado actual de avance sin dar salto de línea automático (end="")
        print(f"Procesando página {pagina} de {paginas_a_descargar}... ", end="")

        # Solicitamos al navegador Chrome acceder a la URL objetivo
        driver.get(url_dinamica)

        # Pausa de seguridad de 4 segundos para permitir la ejecución de scripts de JavaScript (React/Next.js) y renderizado del DOM
        time.sleep(4)

        # Capturamos el código HTML completo ya renderizado por el navegador y se lo pasamos a BeautifulSoup para su análisis
        soup_pag = BeautifulSoup(driver.page_source, 'html.parser')

        # Contador auxiliar para llevar el registro de productos recolectados en la página actual
        conteo_pagina = 0

        # --------------------------------------------------------------------------------------
        # ESTRATEGIA 1: PARSEO DIRECTO DEL OBJETO JSON INTERNO DE NEXT.JS (__NEXT_DATA__)
        # --------------------------------------------------------------------------------------

        # Buscamos en el HTML la etiqueta <script> que contiene el objeto global de estado con id='__NEXT_DATA__'
        next_data_script = soup_pag.find('script', id='__NEXT_DATA__')

        # Verificamos si la etiqueta fue encontrada y contiene datos en formato texto
        if next_data_script and next_data_script.string:
            try:
                # Decodificamos la cadena de texto JSON y la convertimos en un diccionario interactivo de Python
                json_data = json.loads(next_data_script.string)

                # Navegamos en la estructura anidada del objeto hasta llegar a las propiedades de la página
                page_props = json_data.get('props', {}).get('pageProps', {})

                # Intentamos obtener la lista de productos buscando en las claves habituales que usa la API de Falabella
                results = (
                    page_props.get('results') or
                    page_props.get('searchResult', {}).get('results') or
                    page_props.get('products') or []
                )

                # Iteramos sobre cada uno de los productos (bicicletas) encontrados en el arreglo JSON
                for item in results:

                    # Extraemos la referencia / SKU único (probando llaves de respaldo como productId, skuId o id)
                    sku = item.get('productId') or item.get('skuId') or item.get('id', 'No disponible')

                    # Extraemos el nombre comercial del producto
                    nombre = item.get('displayName') or item.get('title') or item.get('name', 'No disponible')

                    # Extraemos la marca o fabricante (ej. GW, Jeep, Oxford, Scoop)
                    marca = item.get('brand') or item.get('brandName') or 'Falabella'

                    # Inicializamos la variable de formato de precio
                    precio_fmt = 'No disponible'

                    # Obtenemos la estructura que contiene la información monetaria del producto
                    prices = item.get('prices', [])

                    # Si existe una lista de precios válida
                    if isinstance(prices, list) and len(prices) > 0:
                        # Extraemos el valor monetario del primer elemento del arreglo de precios
                        p_list = prices[0].get('price', [])
                        if p_list:
                            # Limpiamos puntos y comas de la cadena original para convertirla a número
                            raw_p = str(p_list[0]).replace('.', '').replace(',', '')
                            try:
                                # Damos formato de moneda entera en pesos colombianos (ej: $1,299,900)
                                precio_fmt = f"${float(raw_p):,.0f}"
                            except ValueError:
                                # En caso de no ser numérico, mantenemos el texto original agregando el símbolo $
                                precio_fmt = f"${p_list[0]}"

                    # Si el precio se encuentra directamente en una propiedad llamada 'price'
                    elif 'price' in item:
                        raw_p = str(item['price']).replace('.', '').replace(',', '')
                        try:
                            precio_fmt = f"${float(raw_p):,.0f}"
                        except ValueError:
                            precio_fmt = str(item['price'])

                    # Construimos un diccionario estructurado y lo agregamos a la lista global
                    todos_los_productos.append({
                        "Referencia (SKU)": sku,
                        "Nombre/Marca": f"[{marca}] {nombre}",
                        "Valor": precio_fmt,
                        "Características": f"Bicicletas - Falabella (Pág {pagina})"
                    })

                    # Incrementamos el contador de productos capturados
                    conteo_pagina += 1

            except Exception:
                # Si ocurre alguna excepción procesando el JSON, el flujo continúa de manera silenciosa hacia la estrategia de respaldo
                pass

        # --------------------------------------------------------------------------------------
        # ESTRATEGIA 2: RESPALDO EN DATOS ESTRUCTURADOS SCHEMA.ORG (application/ld+json)
        # --------------------------------------------------------------------------------------

        # Si la estrategia 1 no capturó ningún producto, buscamos en las etiquetas script de tipo Schema.org
        if conteo_pagina == 0:
            scripts_json_ld = soup_pag.find_all('script', type='application/ld+json')

            for script in scripts_json_ld:
                try:
                    if not script.string:
                        continue
                    # Decodificamos el JSON estandarizado de Schema.org
                    datos = json.loads(script.string)

                    # Si el JSON-LD contiene un listado de items (ItemList)
                    if datos.get('@type') == 'ItemList' and 'itemListElement' in datos:
                        for item_wrapper in datos['itemListElement']:
                            producto = item_wrapper.get('item', {})
                            if producto:
                                nombre = producto.get('name', 'No disponible')
                                sku = producto.get('sku', producto.get('id', 'No disponible'))
                                marca_info = producto.get('brand', {})
                                marca = marca_info.get('name', 'Falabella') if isinstance(marca_info, dict) else str(marca_info)

                                ofertas = producto.get('offers', {})
                                precio = 'No disponible'
                                if isinstance(ofertas, dict):
                                    precio_num = ofertas.get('price') or ofertas.get('lowPrice') or ofertas.get('highPrice')
                                    if precio_num:
                                        precio = f"${float(precio_num):,.0f}"

                                todos_los_productos.append({
                                    "Referencia (SKU)": sku,
                                    "Nombre/Marca": f"[{marca}] {nombre}",
                                    "Valor": precio,
                                    "Características": f"Bicicletas - Falabella (Pág {pagina})"
                                })
                                conteo_pagina += 1
                except Exception:
                    continue

        # Imprimimos la confirmación de la cantidad de productos extraídos en la página actual
        print(f"¡Completada! Se encontraron {conteo_pagina} productos.")

finally:
    # Cerramos formalmente la sesión del navegador Selenium y liberamos la memoria del sistema
    driver.quit()


# ==============================================================================================
# 6. PROCESAMIENTO CON PANDAS Y EXPORTACIÓN A ARCHIVO CSV
# ==============================================================================================

# Convertimos la lista global de diccionarios en un DataFrame de Pandas (tabla de datos)
df_paginado = pd.DataFrame(todos_los_productos)

# Verificamos que la tabla contenga datos
if not df_paginado.empty:
    # Eliminamos posibles registros duplicados basándonos en la columna 'Nombre/Marca'
    df_paginado = df_paginado.drop_duplicates(subset=['Nombre/Marca'])

# Imprimimos el total consolidado de bicicletas recolectadas
print(f"\nExtracción finalizada. Total de bicicletas únicas recolectadas: {len(df_paginado)}")

# Si se obtuvieron resultados válidos
if not df_paginado.empty:
    # Muestra los primeros 15 registros de la tabla formateada en pantalla
    display(df_paginado.head(15))

    # Asignamos el nombre del archivo de salida
    nombre_archivo = "productos_falabella_bicicletas.csv"

    # Exportamos el DataFrame a formato CSV usando codificación UTF-8 con BOM ('utf-8-sig') y separador de punto y coma (;) para compatibilidad nativa con Excel
    df_paginado.to_csv(nombre_archivo, index=False, encoding='utf-8-sig', sep=';')
    print(f"Archivo guardado exitosamente como: {nombre_archivo}")

    # Si la ejecución ocurre en Google Colab, dispara la descarga automática del archivo CSV al navegador del usuario
    if EN_COLAB:
        files.download(nombre_archivo)

Iniciando la extracción automatizada de Bicicletas en Falabella...

Procesando página 1 de 5... ¡Completada! Se encontraron 56 productos.
Procesando página 2 de 5... ¡Completada! Se encontraron 54 productos.
Procesando página 3 de 5... ¡Completada! Se encontraron 54 productos.
Procesando página 4 de 5... ¡Completada! Se encontraron 54 productos.
Procesando página 5 de 5... ¡Completada! Se encontraron 54 productos.

Extracción finalizada. Total de bicicletas únicas recolectadas: 217


,Referencia (SKU),Nombre/Marca,Valor,Características
0,154569653,[PROFIT] BICICLETA ELÉCTRICA EON-X 20” MOTOR 5...,"$3,389,900",Bicicletas - Falabella (Pág 1)
1,150829306,[ROCA] Bicicleta Prado 2026 Aluminio Frenos Hd...,"$669,900",Bicicletas - Falabella (Pág 1)
2,73199590,[SCOOP] Bicicleta de montaña modelo modelo Bat...,"$478,990",Bicicletas - Falabella (Pág 1)
3,883542909,[SPIDERMAN] Bicicleta para niños Disney Rin 16...,"$268,990",Bicicletas - Falabella (Pág 1)
4,39275659,[BARBIE] Bicicleta para niñas Rin 16,"$359,990",Bicicletas - Falabella (Pág 1)
7,72068807,[SCOOP] Bicicleta de Montaña Rin 29 Storm,"$649,990",Bicicletas - Falabella (Pág 1)
8,121557837,[GW] NUEVA BICICLETA MTB FALCON 24VEL RIN29 - ...,"$669,900",Bicicletas - Falabella (Pág 1)
9,883542883,[SPIDERMAN] Bicicleta para niños Disney Spidey...,"$238,990",Bicicletas - Falabella (Pág 1)
10,119829958,[ROADMASTER] Bicicleta De Ruta Hurricane Shima...,"$519,900",Bicicletas - Falabella (Pág 1)
11,157221715,[Falabella] Bicicleta Eléctrica 350w Color Menta,"$1,139,900",Bicicletas - Falabella (Pág 1)


Archivo guardado exitosamente como: productos_falabella_bicicletas.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>